# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

RNeNcodec requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (75 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 EnCodec Encoding** - Convert audio to a compressed latent format that RNeNcodec can process
- **📄 Sidecar Creation**- Align parameters with audio frames (75 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "quickstart") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

dataset_path = "../data/quickstartdata/waterfill" # Path to your dataset directory

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [2]:
# Import visualization functions
from dataprep.step_0_visualization import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
quick_analyze(dataset_path, individual_plot_selector=True) # Set individual_plot_selector=True to choose which files to plot.

DATASET SUMMARY:

✅ Found 28 audio-CSV pairs
📁 Parameters: fill_level
⏱️ Total audio duration: 404.0 seconds

FILE DETAILS:

✅ ZOOM0011: 13.2s, 989 frames
⚠️ ZOOM0025: 14.6s, 1093 frames (1 extra audio frames in audio file)
✅ ZOOM0017: 15.0s, 1126 frames
⚠️ ZOOM0001: 14.0s, 1052 frames (1 extra audio frames in audio file)
⚠️ ZOOM0021: 15.1s, 1129 frames (1 extra audio frames in audio file)
⚠️ ZOOM0009: 13.7s, 1031 frames (1 extra audio frames in audio file)
⚠️ ZOOM0012: 13.2s, 993 frames (1 extra audio frames in audio file)
✅ ZOOM0007: 14.4s, 1084 frames
⚠️ ZOOM0006: 14.8s, 1107 frames (1 extra audio frames in audio file)
⚠️ ZOOM0010: 14.2s, 1064 frames (1 extra audio frames in audio file)
✅ ZOOM0015: 14.4s, 1077 frames
⚠️ ZOOM0016: 14.4s, 1076 frames (1 extra audio frames in audio file)
⚠️ ZOOM0023: 15.8s, 1187 frames (1 extra audio frames in audio file)
✅ ZOOM0002: 13.9s, 1042 frames
⚠️ ZOOM0008: 14.7s, 1101 frames (1 extra audio frames in audio file)
⚠️ ZOOM0020: 16.4s, 1231 frames 

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 24kHz mono (required by EnCodec)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset (you may want apply_rms_normalization=False if the relative volume between your dataset files is important).

In [3]:
from dataprep.step_1_normalization import quick_normalize

quick_normalize(dataset_path, apply_rms_normalization=False)


DATA SUMMARY:

Found 28 audio-CSV pairs
RMS normalization: DISABLED (resampling to 24kHz only)
Input: ../data/quickstartdata/waterfill/raw
Output: ../data/quickstartdata/waterfill/normalized

DATA PROCESSING:

Processing: ZOOM0026.wav
    RMS normalization disabled - resampling to 24kHz only
    ✓ Saved: ZOOM0026.wav
    ✅ Copied CSV: ZOOM0026.csv → ../data/quickstartdata/waterfill/normalized/test/ZOOM0026.csv

Processing: ZOOM0027.wav
    RMS normalization disabled - resampling to 24kHz only
    ✓ Saved: ZOOM0027.wav
    ✅ Copied CSV: ZOOM0027.csv → ../data/quickstartdata/waterfill/normalized/test/ZOOM0027.csv

Processing: ZOOM0028.wav
    RMS normalization disabled - resampling to 24kHz only
    ✓ Saved: ZOOM0028.wav
    ✅ Copied CSV: ZOOM0028.csv → ../data/quickstartdata/waterfill/normalized/test/ZOOM0028.csv

Processing: ZOOM0001.wav
    RMS normalization disabled - resampling to 24kHz only
    🔄 Trimmed audio: 1053 → 1052 frames (1 extra audio frames in audio file)
    ✓ Saved: Z

### **3.3 🎵 EnCodec Encoding**

In [4]:
from dataprep.step_2_encodec import quick_encode

quick_encode(dataset_path, bandwidth=6.0, device="cpu")  # Force 8 codebooks and CPU


DATA PROCESSING:

Found 28 WAV files under ../data/quickstartdata/waterfill/normalized
Using device: cpu
Loading EnCodec model (facebook/encodec_24khz)...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The image processor of type `EncodecImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.
Encoding: 100%|███████████████████████████████████████████████████████████████████████████████████| 28/28 [00:08<00:00,  3.40file/s]


DATA SUMMARY:

📁 ZOOM0026.ecdc
   Audio codes shape: (1, 1, 8, 1183)
   Frames: unknown
   Codebooks: unknown
   Original length: 378471 samples

📁 ZOOM0027.ecdc
   Audio codes shape: (1, 1, 8, 1214)
   Frames: unknown
   Codebooks: unknown
   Original length: 388433 samples

📁 ZOOM0028.ecdc
   Audio codes shape: (1, 1, 8, 1085)
   Frames: unknown
   Codebooks: unknown
   Original length: 347148 samples

📁 ZOOM0001.ecdc
   Audio codes shape: (1, 1, 8, 1052)
   Frames: unknown
   Codebooks: unknown
   Original length: 336640 samples

📁 ZOOM0002.ecdc
   Audio codes shape: (1, 1, 8, 1042)
   Frames: unknown
   Codebooks: unknown
   Original length: 333401 samples

📁 ZOOM0003.ecdc
   Audio codes shape: (1, 1, 8, 994)
   Frames: unknown
   Codebooks: unknown
   Original length: 318068 samples

📁 ZOOM0004.ecdc
   Audio codes shape: (1, 1, 8, 965)
   Frames: unknown
   Codebooks: unknown
   Original length: 308800 samples

📁 ZOOM0005.ecdc
   Audio codes shape: (1, 1, 8, 1137)
   Frames: unkn

### **3.4 📄 Sidecar Creation**

In [5]:
from dataprep.step_3_sidecars import quick_create_sidecars

results = quick_create_sidecars(dataset_path)


SIDECAR CREATION:

📁 Tokens directory: ../data/quickstartdata/waterfill/tokens
📁 Raw directory: ../data/quickstartdata/waterfill/raw
📊 Found 28 .ecdc-CSV pairs
🎛️ Parameters: fill_level

SIDECARS SUMMARY:

📄 ZOOM0024 sidecar:
    EnCodec frames: 1042
    CSV annotations: 1042
    ✅ Perfect alignment!
    ✅ Sidecar created (1042 frames, 1 features)

📄 ZOOM0005 sidecar:
    EnCodec frames: 1137
    CSV annotations: 1137
    ✅ Perfect alignment!
    ✅ Sidecar created (1137 frames, 1 features)

📄 ZOOM0019 sidecar:
    EnCodec frames: 1064
    CSV annotations: 1064
    ✅ Perfect alignment!
    ✅ Sidecar created (1064 frames, 1 features)

📄 ZOOM0012 sidecar:
    EnCodec frames: 993
    CSV annotations: 993
    ✅ Perfect alignment!
    ✅ Sidecar created (993 frames, 1 features)

📄 ZOOM0011 sidecar:
    EnCodec frames: 989
    CSV annotations: 989
    ✅ Perfect alignment!
    ✅ Sidecar created (989 frames, 1 features)

📄 ZOOM0009 sidecar:
    EnCodec frames: 1031
    CSV annotations: 1031
   

### **3.5 🤗 HuggingFace Dataset**
(on Windows, you can ignore the "Failed simlink" messages)

In [6]:
from dataprep.step_4_HF import quick_create_dataset

results = quick_create_dataset(dataset_path)


HUGGINGFACE DATASET CREATION:

📁 Tokens directory: ../data/quickstartdata/waterfill/tokens
📁 Output directory: ../data/quickstartdata/waterfill/hf_dataset
🔗 Materialize mode: link

🎯 Split structure detected: train, test
� Creating separate HuggingFace splits for each folder

   📂 train: 25 .ecdc files found
   📂 test: 3 .ecdc files found

📋 Saved expanded conditioning config to: conditioning_config.json
   • Features: fill_level
   • Total features: 1

💾 Saved DatasetDict to: ../data/quickstartdata/waterfill/hf_dataset
📈 Dataset summary:
   • train: 25 samples
   • test: 3 samples

🔍 Verifying dataset files...
✅ All audio files verified successfully
